DATA LOADING

In [1]:
import gdown

file_id = "1D0HCYDfDpz14OOFAyqvnbPUdiIKWHMmD"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "weather.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1D0HCYDfDpz14OOFAyqvnbPUdiIKWHMmD
To: /content/weather.csv
100%|██████████| 1.79M/1.79M [00:00<00:00, 155MB/s]


'weather.csv'

PREPROCESSING

In [2]:
import pandas as pd

df = pd.read_csv("weather.csv")

# Clean column names
df.columns = df.columns.str.replace(" ", "_").str.replace(".", "_")

df.head()

,Data_Precipitation,Date_Full,Date_Month,Date_Week_of,Date_Year,Station_City,Station_Code,Station_Location,Station_State,Data_Temperature_Avg_Temp,Data_Temperature_Max_Temp,Data_Temperature_Min_Temp,Data_Wind_Direction,Data_Wind_Speed
0,0.00,2016-01-03,1,3,2016,Birmingham,BHM,"Birmingham, AL",Alabama,39,46,32,33,4.33
1,0.00,2016-01-03,1,3,2016,Huntsville,HSV,"Huntsville, AL",Alabama,39,47,31,32,3.86
2,0.16,2016-01-03,1,3,2016,Mobile,MOB,"Mobile, AL",Alabama,46,51,41,35,9.73
3,0.00,2016-01-03,1,3,2016,Montgomery,MGM,"Montgomery, AL",Alabama,45,52,38,32,6.86
4,0.01,2016-01-03,1,3,2016,Anchorage,ANC,"Anchorage, AK",Alaska,34,38,29,19,7.80


In [3]:
# Rain classification
df['Rain'] = (df['Data_Precipitation'] > 0).astype(int)

print(df['Rain'].value_counts())

Rain
1    12419
0     4324
Name: count, dtype: int64


In [4]:
features = [
    'Data_Temperature_Avg_Temp',
    'Data_Temperature_Max_Temp',
    'Data_Temperature_Min_Temp',
    'Data_Wind_Speed',
    'Data_Wind_Direction'
]

df = df.dropna()

X = df[features]
y = df['Rain']

TRAIN-TEST SPLIT

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

MODELS


A. Logistic Regression

In [6]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

B. Decision Tree

In [7]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

C. Random Forest

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

EVALUATION

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Decision Tree", y_test, dt_pred)
evaluate("Random Forest", y_test, rf_pred)


Logistic Regression
Accuracy: 0.7563451776649747
Precision: 0.7570900123304563
Recall: 0.9887278582930756
F1 Score: 0.8575418994413407

Decision Tree
Accuracy: 0.772767990444909
Precision: 0.777098745577356
Recall: 0.9726247987117552
F1 Score: 0.863937064187377

Random Forest
Accuracy: 0.7921767691848313
Precision: 0.7981987991994663
Recall: 0.9633655394524959
F1 Score: 0.8730390368478658


In [12]:
importances = rf.feature_importances_

for i, col in enumerate(features):
    print(f"{col}: {importances[i]}")

Data_Temperature_Avg_Temp: 0.15790512370552862
Data_Temperature_Max_Temp: 0.30048008630800893
Data_Temperature_Min_Temp: 0.2249279661166973
Data_Wind_Speed: 0.15921363409964137
Data_Wind_Direction: 0.15747318977012384
